# Notebook 3 — Generalization Study: Does the Truth Direction Transfer?

The central question from the Representation Engineering (RepE) paper is whether there exists a **universal truth direction** inside a language model's activation space. If such a direction exists, then a probe trained on one domain (e.g. self-report honesty) should also detect lies in completely different domains (geography, science, general knowledge).

This distinction matters enormously:

- **If the truth direction transfers across domains**, we have evidence of a genuine, model-wide "honesty signal" — a real lie detector.
- **If it does not transfer**, the probe is merely a dataset-specific classifier that has learned surface-level patterns rather than a fundamental property of the model's representations.

This notebook runs the most important experiment in the project: we **train a probe on `repeng_truthful`** and **evaluate it on all 8 available datasets** — including standard NLP benchmarks (TruthfulQA, ARC, BoolQ) used in the original RepE paper. The degree of transfer tells us how universal the truth encoding is in **Phi-2** (2.7B parameters, 32 layers).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))

import warnings
warnings.filterwarnings("ignore")

from lie_detector_llm.datasets import build_dataset_collection
from lie_detector_llm.experiment import run_transfer_experiment
from lie_detector_llm.plotting import plot_transfer_results

collection = build_dataset_collection()
dataset_names = collection.dataset_names()
print("Available datasets:", dataset_names)

Available datasets: ['cities', 'larger_than', 'qa', 'repeng_truthful']


## Transfer experiment: train on repeng_truthful, evaluate everywhere

We now train a logistic regression (LR) probe on the `repeng_truthful` dataset using activations from the last layer of `gpt2-medium`. We then evaluate that same probe on all four datasets:

- **repeng_truthful** — in-distribution sanity check (should be high)
- **cities** — factual geographic claims (e.g. "Paris is in France")
- **larger_than** — numerical reasoning (e.g. "7 is larger than 3")
- **qa** — general-knowledge question-answer pairs

If the probe truly captures a universal truth direction, it should perform well on all four targets, not just the training domain.

In [ ]:
MODEL = "microsoft/phi-2"

transfer_result = run_transfer_experiment(
    collection=collection,
    train_dataset_name="repeng_truthful",
    eval_dataset_names=dataset_names,
    model_name=MODEL,
    probe_method="lr",
    layer_index=-1,
)
print(transfer_result.summary_table().to_string(index=False))

In [ ]:
fig, ax = plot_transfer_results(transfer_result.results, title="Transfer: trained on repeng_truthful (Phi-2)")
fig

## Probe comparison: do different methods capture different aspects of truth?

Not all probes look for the same thing. The four methods we compare capture the truth direction in different ways:

- **DIM** (Difference-in-Means): takes the mean activation of true statements minus false statements — the simplest geometric approach.
- **LAT** (Linear Artificial Tomography): fits a linear separator in the activation space.
- **LR** (Logistic Regression): a supervised classifier trained on labeled activations.
- **PCA-G** (PCA on grouped differences): extracts the principal component of the true/false difference vectors.

DIM and LAT are purely geometric (unsupervised), while LR is supervised. Comparing their transfer performance tells us whether the truth direction is a robust geometric property or something that requires supervision to extract.

In [4]:
import pandas as pd

all_results = []
for method in ["dim", "lat", "lr", "pca-g"]:
    out = run_transfer_experiment(
        collection=collection,
        train_dataset_name="repeng_truthful",
        eval_dataset_names=dataset_names,
        model_name=MODEL,
        probe_method=method,
        layer_index=-1,
    )
    all_results.append(out.summary_table())

comparison = pd.concat(all_results, ignore_index=True)
pivot = comparison.pivot_table(index="probe_method", columns="eval_dataset", values="grouped_accuracy")
print(pivot.round(2).to_string())

eval_dataset  cities  larger_than    qa  repeng_truthful
probe_method                                            
dim             0.47         0.53  0.23             0.61
lat             0.43         0.53  0.10             0.46
lr              0.40         0.53  0.30             0.94
pca-g           0.43         0.53  0.10             0.46


## Interpretation

The pivot table above answers three key questions:

1. **Which probe transfers best?** Compare the off-diagonal accuracies (cities, larger_than, qa) across probe methods. A method that scores well across all evaluation datasets — not just on repeng_truthful — is the one that best captures a transferable truth direction.

2. **Which target datasets are easiest/hardest?** Some domains may align more naturally with the training distribution. For instance, `larger_than` (numerical reasoning) may be structurally different from factual claims in `repeng_truthful`, making it a harder transfer target. Conversely, `qa` may share more topical overlap.

3. **What does this tell us about universality?** If all probes transfer well, the truth direction in `gpt2-medium` is a robust, domain-independent feature of the model's representations — strong evidence for the RepE hypothesis. If transfer is poor or probe-dependent, truth encoding may be more localized or entangled with domain-specific patterns, and the "lie detector" framing becomes harder to justify.